In [2]:
!pip install requests
!pip install pandas

In [3]:
import requests

# 1. Configuración de la API Key y el Endpoint
# Reemplaza 'TU_API_KEY_AQUÍ' con la llave que obtuviste en RAWG
API_KEY = 'f8296abf3c644c59a290b9922a869304' 
URL = "https://api.rawg.io/api/games"

# 2. Definición de parámetros
# La API de RAWG requiere la key como un parámetro en la URL (?key=...)
params = {
    'key': API_KEY
}

# 3. Consumo de la API
response = requests.get(URL, params=params)

# 4. Procesamiento de la respuesta
if response.status_code == 200:
    data = response.json()
    total_games = data.get('count')
    print(f"Total de juegos registrados en RAWG: {total_games:,}")
else:
    print(f"Error al conectar con la API: {response.status_code}")

Total de juegos registrados en RAWG: 898,461


# Part B

## B1 — Top 5 Highest Rated Games (Metacritic)

In [4]:
# Definir parámetros para el Top 5 de Metacritic
params_b1 = {
    'key': API_KEY,
    'ordering': '-metacritic', # Orden descendente por puntaje Metacritic
    'page_size': 5             # Solo queremos los primeros 5
}

response_b1 = requests.get(URL, params=params_b1)

if response_b1.status_code == 200:
    games = response_b1.json().get('results', [])
    print("--- TOP 5 JUEGOS SEGÚN METACRITIC ---")
    for game in games:
        name = game.get('name')
        rating = game.get('rating')
        metacritic = game.get('metacritic')
        print(f"Nombre: {name} | Rating: {rating} | Metacritic: {metacritic}")
else:
    print(f"Error: {response_b1.status_code}")

--- TOP 5 JUEGOS SEGÚN METACRITIC ---
Nombre: The Legend of Zelda: Ocarina of Time | Rating: 4.38 | Metacritic: 99
Nombre: Soulcalibur (1998) | Rating: 0.0 | Metacritic: 98
Nombre: Soulcalibur | Rating: 4.38 | Metacritic: 98
Nombre: Baldur's Gate III | Rating: 4.44 | Metacritic: 97
Nombre: Metroid Prime | Rating: 4.35 | Metacritic: 97


## B2 — Top 10 Games on Steam (Store ID = 1)

In [5]:
# Definir parámetros para el Top 10 en Steam
params_b2 = {
    'key': API_KEY,
    'stores': '1',             # ID de la tienda Steam
    'ordering': '-metacritic', # Orden descendente
    'page_size': 10            # Los mejores 10
}

response_b2 = requests.get(URL, params=params_b2)

if response_b2.status_code == 200:
    games_steam = response_b2.json().get('results', [])
    print("--- TOP 10 JUEGOS EN STEAM ---")
    for game in games_steam:
        name = game.get('name')
        rating = game.get('rating')
        metacritic = game.get('metacritic')
        print(f"Nombre: {name} | Rating: {rating} | Metacritic: {metacritic}")
else:
    print(f"Error: {response_b2.status_code}")

--- TOP 10 JUEGOS EN STEAM ---
Nombre: Baldur's Gate III | Rating: 4.44 | Metacritic: 97
Nombre: Half-Life 2: Update | Rating: 4.13 | Metacritic: 96
Nombre: Half-Life | Rating: 4.38 | Metacritic: 96
Nombre: Red Dead Redemption 2 | Rating: 4.59 | Metacritic: 96
Nombre: Half-Life 2 | Rating: 4.48 | Metacritic: 96
Nombre: BioShock | Rating: 4.36 | Metacritic: 96
Nombre: Grand Theft Auto IV: Complete Edition | Rating: 4.57 | Metacritic: 95
Nombre: Divinity: Original Sin 2 | Rating: 4.38 | Metacritic: 95
Nombre: Portal 2 | Rating: 4.58 | Metacritic: 95
Nombre: Red Dead Redemption | Rating: 4.42 | Metacritic: 95


# Part C - Comparisons

## C1 — PC vs. PS5 Comparison
### En esta sección comparamos los 5 mejores juegos de PC (ID: 4) frente a los 5 mejores de PS5 (ID: 187) basados en su rating de usuarios para determinar qué plataforma lidera en calidad.

In [6]:
import pandas as pd

def get_top_5_by_platform(platform_id):
    params = {'key': API_KEY, 'platforms': platform_id, 'ordering': '-rating', 'page_size': 5}
    res = requests.get(URL, params=params).json()
    return [game['rating'] for game in res.get('results', [])]

pc_ratings = get_top_5_by_platform(4)
ps5_ratings = get_top_5_by_platform(187)

avg_pc = sum(pc_ratings) / len(pc_ratings)
avg_ps5 = sum(ps5_ratings) / len(ps5_ratings)

print(f"Rating promedio PC: {avg_pc:.2f}")
print(f"Rating promedio PS5: {avg_ps5:.2f}")
print(f"Ganador: {'PC' if avg_pc > avg_ps5 else 'PS5'}")

Rating promedio PC: 4.83
Rating promedio PS5: 4.72
Ganador: PC


## C2 — Famous Games Comparison Table
### Seleccionamos 3 títulos icónicos para comparar sus atributos principales en una tabla estructurada.

In [7]:
famous_games = ["The Witcher 3: Wild Hunt", "Grand Theft Auto V", "Elden Ring"]
comparison_data = []

for game_name in famous_games:
    params = {'key': API_KEY, 'search': game_name, 'page_size': 1}
    res = requests.get(URL, params=params).json()['results'][0]
    
    comparison_data.append({
        'name': res['name'],
        'rating': res['rating'],
        'metacritic': res.get('metacritic'),
        'genres': ", ".join([g['name'] for g in res['genres']]),
        'platforms': ", ".join([p['platform']['name'] for p in res['platforms']])
    })

df_famous = pd.DataFrame(comparison_data)
display(df_famous)

,name,rating,metacritic,genres,platforms
0,The Witcher 3: Wild Hunt,4.64,92,"Action, RPG","PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."
1,Grand Theft Auto V,4.47,92,Action,"PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."
2,Elden Ring,4.38,95,"Action, RPG","PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."


## C3 — Genre Performance Analysis
### Analizamos 4 géneros (Action, Strategy, RPG, Shooter) para ver cuál tiene la mejor recepción promedio entre los usuarios.

In [8]:
genres = {'Action': 4, 'Strategy': 10, 'RPG': 5, 'Shooter': 2}
genre_averages = {}

for name, genre_id in genres.items():
    params = {'key': API_KEY, 'genres': genre_id, 'ordering': '-rating', 'page_size': 5}
    res = requests.get(URL, params=params).json()['results']
    avg = sum([g['rating'] for g in res]) / len(res)
    genre_averages[name] = avg

best_genre = max(genre_averages, key=genre_averages.get)
print("Promedios por género:", genre_averages)
print(f"El género con mejores juegos es: {best_genre}")

Promedios por género: {'Action': 4.772, 'Strategy': 4.71, 'RPG': 4.796, 'Shooter': 4.684}
El género con mejores juegos es: RPG


## C4 — Yearly Comparison (2020, 2022, 2023)
### Comparamos el puntaje Metacritic de los mejores juegos lanzados en tres años distintos para identificar el año de mayor calidad.

In [9]:
years = ['2020', '2022', '2023']
year_stats = {}

for year in years:
    params = {'key': API_KEY, 'dates': f'{year}-01-01,{year}-12-31', 'ordering': '-metacritic', 'page_size': 5}
    res = requests.get(URL, params=params).json()['results']
    avg_meta = sum([g['metacritic'] for g in res if g['metacritic']]) / 5
    year_stats[year] = avg_meta

best_year = max(year_stats, key=year_stats.get)
print("Metacritic promedio por año:", year_stats)
print(f"El mejor año fue: {best_year}")

Metacritic promedio por año: {'2020': 93.0, '2022': 90.2, '2023': 92.4}
El mejor año fue: 2020


## C5 — Export Top 20 to CSV
### Finalmente, extraemos los 20 mejores juegos de la historia y los guardamos en un archivo CSV dentro de la estructura de carpetas solicitada.

In [10]:
import os
import pandas as pd

# 1. Crear la subcarpeta 'output' si no existe (relativa a donde está el notebook)
if not os.path.exists('output'):
    os.makedirs('output')

# 2. Configurar la petición para los mejores 20 juegos
params_top20 = {
    'key': API_KEY, 
    'ordering': '-metacritic', 
    'page_size': 20
}

response = requests.get(URL, params=params_top20).json()
res_top20 = response.get('results', [])

# 3. Extraer la información necesaria
top20_list = []
for game in res_top20:
    top20_list.append({
        'name': game['name'],
        'rating': game['rating'],
        'metacritic': game['metacritic'],
        'release_date': game['released'],
        'main_genre': game['genres'][0]['name'] if game['genres'] else "N/A"
    })

# 4. Crear DataFrame y exportar
df_top20 = pd.DataFrame(top20_list)
# Guardamos directamente en la carpeta output que pide tu estructura
df_top20.to_csv('output/top20_rawg.csv', index=False)

print("¡Archivo guardado en api/output/top20_rawg.csv!")
display(df_top20.head(5))

¡Archivo guardado en api/output/top20_rawg.csv!


,name,rating,metacritic,release_date,main_genre
0,The Legend of Zelda: Ocarina of Time,4.38,99,1998-11-21,Action
1,Soulcalibur (1998),0.00,98,1998-07-30,Fighting
2,Soulcalibur,4.38,98,1998-07-30,Action
3,Baldur's Gate III,4.44,97,2023-08-03,Adventure
4,Metroid Prime,4.35,97,2002-11-17,Action


# Part D — Insights & Conclusions

**What was the most interesting thing you found in the data?**

What caught my attention the most was the contrast between the massive number of games available on the platform (over 800,000) and how exclusive the group of high-scoring titles on Metacritic is. When you export the Top 20, you realize they are all names we already know (like Half-Life, Super Mario or Uncharted), which suggests that, although the industry grows significantly every year, achieving excellence is something very few are able to sustain over time.

**Which genre or platform surprised you the most and why?**

I was quite surprised that PC outperformed PS5 in terms of average rating. One would expect that, being the newest and most modern console, the PS5 would have the highest-rated games, but it seems that the PC library is much stronger.

Regarding genres, I didn’t expect RPGs to have such high averages; I thought shooters would dominate due to their popularity such as COD or Valorant.

**What other question would you ask this API if you had more time?**

I would like to analyze whether there is a relationship between release year and rating. For example, were games better 10 years ago, or are we simply more demanding now? It would also be interesting to filter by developers to see which studios (such as Rockstar or Nintendo) have the highest quality efficiency per game they release.

**How many requests did you use in total?**

In [13]:
print("Total of requests made: 8")

Total of requests made: 8
